# 🏠 Magicbricks Properties — Complete Project Notebook
### Web Scraping → EDA → Advanced Statistics → Price Prediction ML Pipeline

**Dataset:** ~5,000 Indian property listings across 6 cities
**Target:** Predict property price from features (BHK, Area, City, Furnishing, Status)
**Best Algorithm:** Gradient Boosting Regressor (tuned via GridSearch · RandomSearch · Bayesian Optuna)

---
| Part | Content |
|------|---------|
| **Part 1** | Web Scraper — collect listings from Magicbricks |
| **Part 2** | Data Loading & Cleaning |
| **Part 3** | EDA — Univariate & Bivariate Analysis |
| **Part 4** | Advanced Statistics (Basic Stats · Distributions · Outliers · Hypothesis Tests) |
| **Part 5** | ML Pipeline — Preprocessing · Feature Selection · Hyperparameter Tuning · CV · Bias-Variance |

# PART 1 — Web Scraper
> Collect ~5,000-10,000 property listings from Magicbricks across multiple cities.

In [ ]:
import requests                  # downloads web pages
from bs4 import BeautifulSoup    # reads the HTML of a page
import pandas as pd              # makes the data table
import numpy as np               # gives us "empty value" (np.nan)
import re                        # regex: finds details inside text
import time                      # lets us pause between pages

## Step 1 — Settings

Just three things to set: the cities, how many pages to scrape, and a browser label so the site replies normally.

In [ ]:
# Cities to search. More cities = more rows AND a more interesting EDA
CITIES = ["Hyderabad", "Bengaluru", "Pune", "Chennai",
          "Mumbai", "Delhi", "Gurgaon", "Noida"]

PAGES_PER_CITY = 40      # pages to check per city (stops early when city is finished)
MAX_RECORDS    = 10000   # stop once we have this many houses
SLEEP_SECONDS  = 2       # pause between pages so we don't get blocked

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "                  "AppleWebKit/537.36 (KHTML, like Gecko) "                  "Chrome/114.0.0.0 Safari/537.36"
}

URL_TEMPLATE = (
    "https://www.magicbricks.com/property-for-sale-rent-in-{city}/"    "residential-real-estate-{city}?page={page}"
)
CITY = CITIES[0]   # default city for single-page test

## Step 2 — Test one page first

Always check one page before scraping many. About **30 cards** means it works ✅

In [ ]:
url       = URL_TEMPLATE.format(city=CITY, page=1)
page_data = requests.get(url, headers=HEADERS)
soup      = BeautifulSoup(page_data.text, "html.parser")

cards = soup.find_all("div", class_="mb-srp__card")   # each "card" is one house

print("status code :", page_data.status_code)   # 200 = OK
print("cards found :", len(cards))

if len(cards) > 0:
    print("\n----- text of the first house -----")
    print(cards[0].get_text(separator=" | ", strip=True)[:500])

## Step 3 — A quick look at regex

`re.search(pattern, text)` finds the first match. We read the part inside the brackets `( )` with `.group(1)`. If nothing is found we store `np.nan`.

| Symbol | Meaning |
|---|---|
| `\d` | a digit |
| `+` | one or more |
| `\s` | a space |
| `*` | zero or more |
| `( )` | the part we keep |

## Step 4 — Scrape the houses

Plan: make empty lists → loop over each city and page → read every card → save each detail into its list. One house = one new value in every list.

In [ ]:
# one empty list for each detail we want to collect
city_list, title_list, society_list = [], [], []
bhk_list, area_list, bath_list      = [], [], []
furnish_list, status_list           = [], []
age_list, price_list, link_list     = [], [], []

seen = set()   # remembers houses already saved, so we skip repeats

for city in CITIES:
    for page in range(1, PAGES_PER_CITY + 1):

        if len(price_list) >= MAX_RECORDS:
            break

        url       = URL_TEMPLATE.format(city=city, page=page)
        page_data = requests.get(url, headers=HEADERS)
        soup      = BeautifulSoup(page_data.text, "html.parser")
        cards     = soup.find_all("div", class_="mb-srp__card")

        if not cards:
            break   # no cards → this city is finished

        new_this_page = 0
        for card in cards:
            text = card.get_text(separator=" ", strip=True)
            link = card.find("a", href=True)
            link = "https://www.magicbricks.com" + link["href"] if link else np.nan

            if link in seen:
                continue
            seen.add(link)
            new_this_page += 1

            def grab(pattern, text=text):
                m = re.search(pattern, text, re.IGNORECASE)
                return m.group(1).strip() if m else np.nan

            bhk   = grab(r"(\d+(?:\.\d+)?)\s*BHK")
            area  = grab(r"(\d+(?:,\d+)?)\s*(?:sq\.?ft|sqft)")
            bath  = grab(r"(\d+)\s*Baths?")
            fur   = grab(r"(Furnished|Unfurnished|Semi-Furnished)")
            stat  = grab(r"(Ready to Move|Under Construction|New Launch)")
            age   = grab(r"(\d+\s*(?:year|yr)s?\s*old|New Property|Under Construction)")
            price = grab(r"(?:₹|Rs\.?)\s?([\d,\.]+\s*(?:Lac|Cr|L|Cr\.)?)")

            city_list.append(city)
            title_list.append(card.find("h2") and card.find("h2").get_text(strip=True) or np.nan)
            soc = card.find("div", class_="mb-srp__card--society")
            society_list.append(soc.get_text(strip=True) if soc else np.nan)
            bhk_list.append(float(bhk.replace(',','')) if bhk and bhk != np.nan else np.nan)
            area_list.append(float(str(area).replace(',','')) if area and area != np.nan else np.nan)
            bath_list.append(float(bath) if bath and bath != np.nan else np.nan)
            furnish_list.append(fur)
            status_list.append(stat)
            age_list.append(age)
            price_list.append(price)
            link_list.append(link)

        if new_this_page == 0:
            break   # all new cards seen → city finished

        time.sleep(SLEEP_SECONDS)

    if len(price_list) >= MAX_RECORDS:
        break

print(f"Scraped {len(price_list):,} listings from {len(seen):,} unique URLs")

## Step 5 — Make the table and save it

In [ ]:
data = {
    "City":        city_list,
    "Title":       title_list,
    "Society":     society_list,
    "BHK":         bhk_list,
    "Area_sqft":   area_list,
    "Bathrooms":   bath_list,
    "Furnishing":  furnish_list,
    "Status":      status_list,
    "Age":         age_list,
    "Price_INR":   price_list,
    "Listing_URL": link_list,
}

mb_df = pd.DataFrame(data)
mb_df = mb_df.drop_duplicates(subset="Listing_URL").reset_index(drop=True)

print("Table size (rows, columns):", mb_df.shape)
mb_df.head()

In [ ]:
mb_df.to_csv("magicbricks_properties.csv", index=False)
print("Saved: magicbricks_properties.csv  with", len(mb_df), "houses")

## How to explain this project in an interview

**One-line summary:**
> "I built a web scraper that collects ~5,000–10,000 property listings from Magicbricks across several cities and turns them into a clean dataset — covering price, size, BHK, furnishing, status and age — ready for EDA and price prediction modelling."

**Walk through in 5 steps:**
1. Send a `requests.get()` with a browser User-Agent so the site responds normally.
2. Parse HTML with `BeautifulSoup`, grab every property card with `find_all`.
3. Extract details from each card's text using `re.search` (regex).
4. Store everything in parallel lists → build a `pandas` DataFrame.
5. Deduplicate by URL and save as CSV.

# PART 2 — Data Loading & Cleaning

> The `.xls` file is actually a CSV — read with `pd.read_csv`. Real scraped data is messy; we fix it step by step.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import (norm, skew, kurtosis,
                          ttest_ind, f_oneway, chi2_contingency,
                          shapiro, kstest, probplot)
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

COLORS = ['#4C8BF5', '#FF6B6B', '#34C98A', '#FF9F43', '#A685E2', '#5BC8F5']

In [ ]:
df = pd.read_csv(r"C:\Users\vishwa\Downloads\magicbricks_properties.xls")

In [ ]:
print("rows and columns:", df.shape)
df.head()

## Data Cleaning

Step by step:
1. Column names → lowercase
2. Text values → lowercase + trim spaces
3. Drop `Bathrooms` (99% missing, unreliable values)
4. Remove rows with no area or no price
5. Keep sensible home sizes (200–10000 sqft) and prices (₹5L–₹50Cr)
6. Add `price_per_sqft` and `price_cr` derived columns

In [ ]:
# 1) column names -> lowercase
df.columns = df.columns.str.lower()

# 2) text values -> lowercase + trim
text_columns = ["city", "title", "society", "furnishing", "status", "age", "listing_url"]
for col in text_columns:
    df[col] = df[col].str.strip().str.lower()

# 3) drop unusable bathrooms column
df = df.drop(columns=["bathrooms"])

print("columns now:", list(df.columns))
df.head()

In [ ]:
print("rows before cleaning:", len(df))

# 4) remove rows with no area or no price
df = df.dropna(subset=["area_sqft", "price_inr"])

# 5) keep only sensible homes
df = df[(df["area_sqft"] >= 200) & (df["area_sqft"] <= 10000)]
df = df[(df["price_inr"] >= 500_000) & (df["price_inr"] <= 500_000_000)]

# 6) add derived columns
df["price_per_sqft"] = df["price_inr"] / df["area_sqft"]
df = df[(df["price_per_sqft"] >= 1000) & (df["price_per_sqft"] <= 100_000)]
df["price_cr"] = df["price_inr"] / 10_000_000

print("rows after cleaning :", len(df))
print("\nmissing values per column:")
print(df.isna().sum())

In [ ]:
df.info()

# PART 3 — Exploratory Data Analysis

## 3.1 Univariate Analysis — Numbers

In [ ]:
df[["bhk", "area_sqft", "price_inr", "price_per_sqft"]].describe().round(2)

In [ ]:
print("PRICE (in crores):")
print("  mean   (average)      :", round(df["price_cr"].mean(), 2))
print("  median (middle value) :", round(df["price_cr"].median(), 2))
print("  std    (spread)       :", round(df["price_cr"].std(), 2))

print("\nAREA (sqft):")
print("  mean   :", round(df["area_sqft"].mean()))
print("  median :", round(df["area_sqft"].median()))

print("\nMost common BHK (mode):", int(df["bhk"].mode()[0]))

> 💡 The **mean price is higher than the median** — a few very expensive homes pull the average up. The median is the more honest "typical" value for right-skewed data.

In [ ]:
for col in ["city", "furnishing", "status"]:
    print("----", col, "----")
    print(df[col].value_counts())
    print()

In [ ]:
(df["city"].value_counts(normalize=True) * 100).round(1)

## 3.2 Bivariate Analysis — Numbers

In [ ]:
city_summary = (df.groupby("city")[["price_cr", "price_per_sqft"]]
                  .median().round(2)
                  .sort_values("price_cr", ascending=False))
city_summary

In [ ]:
homes = df[df["bhk"].between(1, 6)]
homes.groupby("bhk")[["price_cr", "area_sqft"]].mean().round(2)

In [ ]:
df.groupby("furnishing")["price_cr"].median().round(2).sort_values(ascending=False)

In [ ]:
df[["bhk", "area_sqft", "price_inr", "price_per_sqft"]].corr().round(2)

In [ ]:
pd.crosstab(df["city"], df["status"])

## 3.3 Univariate Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Price
sns.histplot(df["price_cr"], bins=50, ax=axes[0], color=COLORS[0])
axes[0].set_title("Price Distribution (₹ Crores)")
axes[0].set_xlabel("price (crores)")

# Area
sns.histplot(df["area_sqft"], bins=50, ax=axes[1], color=COLORS[2])
axes[1].set_title("Area Distribution (sqft)")
axes[1].set_xlabel("area (sqft)")

# Price boxplot
axes[2].boxplot(df["price_cr"].values, vert=True, patch_artist=True,
                boxprops=dict(facecolor=COLORS[0], alpha=0.6),
                flierprops=dict(marker='.', color=COLORS[1], markersize=3))
axes[2].set_title("Price Spread & Outliers")
axes[2].set_ylabel("price (crores)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.countplot(data=df, x="city", ax=axes[0],
              order=df["city"].value_counts().index, palette=COLORS)
axes[0].set_title("Homes per City")
axes[0].tick_params(axis="x", rotation=45)

homes = df[df["bhk"].between(1, 6)]
sns.countplot(data=homes, x="bhk", ax=axes[1], palette=COLORS)
axes[1].set_title("Homes per BHK")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="furnishing", ax=axes[0],
              order=df["furnishing"].value_counts().index, palette=COLORS)
axes[0].set_title("Furnishing Type")
axes[0].tick_params(axis="x", rotation=20)

sns.countplot(data=df, x="status", ax=axes[1],
              order=df["status"].value_counts().index, palette=COLORS)
axes[1].set_title("Property Status")

plt.tight_layout()
plt.show()

## 3.4 Bivariate Charts

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="area_sqft", y="price_cr", alpha=0.25, color=COLORS[0])
plt.title("Area vs Price (each dot = one home)")
plt.xlabel("area (sqft)")
plt.ylabel("price (crores)")
plt.show()

In [ ]:
order = df.groupby("city")["price_cr"].median().sort_values(ascending=False).index
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="city", y="price_cr", order=order, palette=COLORS)
plt.title("Price by City")
plt.xlabel("")
plt.ylabel("price (crores)")
plt.ylim(0, 8)
plt.xticks(rotation=45)
plt.show()

In [ ]:
homes = df[df["bhk"].between(1, 6)]
plt.figure(figsize=(8, 5))
sns.boxplot(data=homes, x="bhk", y="price_cr", palette=COLORS)
plt.title("Price by BHK")
plt.xlabel("BHK (bedrooms)")
plt.ylabel("price (crores)")
plt.ylim(0, 8)
plt.show()

In [ ]:
order = df.groupby("city")["price_per_sqft"].median().sort_values(ascending=False).index
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x="city", y="price_per_sqft", order=order,
            estimator=np.median, errorbar=None, palette=COLORS)
plt.title("Median Price per sqft by City")
plt.xlabel("")
plt.ylabel("₹ per sqft")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
corr = df[["bhk", "area_sqft", "price_inr", "price_per_sqft"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f",
            linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

# PART 4 — Advanced Statistics

| Section | Topic |
|---------|-------|
| 4.1 | Basic Statistics — Mean, Median, Mode, Variance, CV |
| 4.2 | Five-Number Summary & IQR |
| 4.3 | Skewness & Kurtosis |
| 4.4 | Outlier Detection (IQR · Z-Score · Modified Z-Score) |
| 4.5 | Distribution Fitting (Normal · Log-Normal) |
| 4.6 | Hypothesis Testing (t-test · ANOVA · Chi-Square) |

## 4.1 Basic Statistics

In [ ]:
price = df['price_cr'].dropna()
area  = df['area_sqft'].dropna()

print("=" * 45)
print("💰 PRICE STATISTICS  (in ₹ Crores)")
print("=" * 45)
print(f"  Mean (Average)   : ₹{price.mean():.2f} Cr")
print(f"  Median (Middle)  : ₹{price.median():.2f} Cr")
print(f"  Mode (Most Common): ₹{price.mode()[0]:.2f} Cr")
print(f"  Variance         : {price.var():.4f}")
print(f"  Std Dev          : ₹{price.std():.2f} Cr")
print(f"  CV (Spread%)     : {(price.std()/price.mean())*100:.1f}%")
print()
print("=" * 45)
print("📐 AREA STATISTICS  (Square Feet)")
print("=" * 45)
print(f"  Mean   : {area.mean():.1f} sqft")
print(f"  Median : {area.median():.0f} sqft")
print(f"  Std Dev: {area.std():.1f} sqft")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

price_clipped = price[price <= 20]
price_clipped.plot.hist(bins=60, ax=ax, color=COLORS[0], edgecolor='white',
                         alpha=0.85, label='Price distribution')

for val, label, color in [
    (price.mean(),   f'Mean  ₹{price.mean():.1f}Cr',   COLORS[1]),
    (price.median(), f'Median ₹{price.median():.1f}Cr', COLORS[2]),
    (price.mode()[0],f'Mode  ₹{price.mode()[0]:.1f}Cr', COLORS[3]),
]:
    ax.axvline(val, color=color, linewidth=2.5, linestyle='--', label=label)

ax.set_title('Mean vs Median vs Mode — Price (₹ Crores)')
ax.set_xlabel('Price (₹ Crores)')
ax.legend()
plt.tight_layout()
plt.show()

## 4.2 Five-Number Summary & IQR

In [ ]:
def five_num_summary(series, name, unit=''):
    q1 = series.quantile(0.25); q3 = series.quantile(0.75)
    iqr = q3 - q1
    print(f"\n📦 Five-Number Summary — {name}")
    print("-" * 40)
    print(f"  Min          : {series.min():.2f} {unit}")
    print(f"  Q1  (25%)    : {q1:.2f} {unit}")
    print(f"  Median (50%) : {series.median():.2f} {unit}")
    print(f"  Q3  (75%)    : {q3:.2f} {unit}")
    print(f"  Max          : {series.max():.2f} {unit}")
    print(f"  IQR          : {iqr:.2f} {unit}")
    print(f"  Lower Fence  : {q1-1.5*iqr:.2f} {unit}")
    print(f"  Upper Fence  : {q3+1.5*iqr:.2f} {unit}")

five_num_summary(price, 'Price', '₹ Cr')
five_num_summary(area,  'Area', 'sqft')

In [ ]:
pcts = [10, 25, 50, 75, 90, 95, 99]
print("📏 Price Percentile Table")
print("-" * 35)
for p in pcts:
    val = price.quantile(p/100)
    bar = '█' * int(p / 10)
    print(f"  {p:3d}th percentile  ₹{val:.2f} Cr  {bar}")

## 4.3 Skewness & Kurtosis

In [ ]:
print("=" * 55)
print(f"{'Column':<18} {'Skewness':>10} {'Kurtosis':>10}  Shape")
print("=" * 55)

for col in ['price_cr', 'area_sqft', 'bhk', 'price_per_sqft']:
    s  = df[col].dropna()
    sk = skew(s); ku = kurtosis(s, fisher=True)
    shape = ("➡️ Right Skewed" if sk > 1 else
             "↗️ Mild Right"  if sk > 0 else
             "⬅️ Left Skewed" if sk < -1 else
             "↙️ Mild Left")
    print(f"  {col:<18} {sk:>10.2f} {ku:>10.2f}  {shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

price_cap = price[price <= 15]
axes[0].hist(price_cap, bins=50, density=True, color=COLORS[0],
             edgecolor='white', alpha=0.7, label='Actual data')
x_line = np.linspace(0, 15, 300)
axes[0].plot(x_line, norm.pdf(x_line, price_cap.mean(), price_cap.std()),
             'r--', lw=2.5, label='Normal fit')
axes[0].set_title(f'Price: Skew={skew(price_cap):.2f}, Kurt={kurtosis(price_cap):.2f}')
axes[0].set_xlabel('Price (₹ Cr)'); axes[0].legend()

log_price = np.log1p(price)
axes[1].hist(log_price, bins=50, density=True, color=COLORS[2],
             edgecolor='white', alpha=0.7, label='log(price)')
axes[1].plot(np.linspace(log_price.min(), log_price.max(), 200),
             norm.pdf(np.linspace(log_price.min(), log_price.max(), 200),
                      log_price.mean(), log_price.std()),
             'r--', lw=2.5, label='Normal fit')
axes[1].set_title(f'log(Price): Skew={skew(log_price):.2f} → much more symmetric!')
axes[1].set_xlabel('log(Price)'); axes[1].legend()

plt.tight_layout()
plt.show()

## 4.4 Outlier Detection

In [ ]:
# Method 1: IQR Fence
q1, q3 = price.quantile(0.25), price.quantile(0.75)
iqr     = q3 - q1
lf, uf  = q1 - 1.5*iqr, q3 + 1.5*iqr
iqr_out = df[(df['price_cr'] < lf) | (df['price_cr'] > uf)]

print("📦 IQR Method")
print(f"   Lower Fence : ₹{lf:.2f} Cr")
print(f"   Upper Fence : ₹{uf:.2f} Cr")
print(f"   Outliers    : {len(iqr_out):,} ({len(iqr_out)/len(df)*100:.1f}%)")
print()

# Method 2: Z-Score
z_scores   = np.abs(stats.zscore(price))
z_out_cnt  = (z_scores > 3).sum()
print("📊 Z-Score Method (|z| > 3)")
print(f"   Outliers : {z_out_cnt:,} ({z_out_cnt/len(price)*100:.1f}%)")
print()

# Method 3: Modified Z-Score (Median Absolute Deviation)
mad        = np.median(np.abs(price - price.median()))
mod_z      = 0.6745 * (price - price.median()) / mad
mod_out    = (np.abs(mod_z) > 3.5).sum()
print("🔧 Modified Z-Score Method (|mz| > 3.5)")
print(f"   Outliers : {mod_out:,} ({mod_out/len(price)*100:.1f}%)")

## 4.5 Distribution Fitting

In [ ]:
sample     = price.sample(min(500, len(price)), random_state=42)
stat_sw, p_sw = shapiro(sample)

print("🔬 Shapiro-Wilk Normality Test")
print(f"   W : {stat_sw:.4f}   p-value : {p_sw:.6f}")
if p_sw < 0.05:
    print("   ❌ p < 0.05 → REJECT normality → Price is NOT normally distributed")
else:
    print("   ✅ p ≥ 0.05 → Could be normal")

# Log-normal test
log_price = np.log1p(price.clip(lower=0.01))
stat_ln, p_ln = shapiro(log_price.sample(500, random_state=42))
print(f"\n🔬 Shapiro-Wilk on log(Price): p = {p_ln:.4f}")
print("   → log(Price) is closer to normal — supports log-transform for ML")

## 4.6 Hypothesis Testing

In [ ]:
# Test 1: t-test — Mumbai vs Chennai
mumbai  = df[df['city']=='mumbai']['price_cr'].dropna()
chennai = df[df['city']=='chennai']['price_cr'].dropna()
t_stat, p_val = ttest_ind(mumbai, chennai, equal_var=False)

print("=" * 52)
print("🏙️  Test 1: Welch t-test — Mumbai vs Chennai")
print("=" * 52)
print(f"   Mumbai  : mean=₹{mumbai.mean():.2f}Cr  n={len(mumbai)}")
print(f"   Chennai : mean=₹{chennai.mean():.2f}Cr  n={len(chennai)}")
print(f"   t-stat  : {t_stat:.4f}   p-value: {p_val:.6f}")
print("   ✅ REJECT H₀" if p_val < 0.05 else "   ✗ FAIL to reject H₀")

In [ ]:
# Test 2: ANOVA — all 6 cities
city_groups = [df[df['city']==c]['price_cr'].dropna() for c in df['city'].unique()]
f_stat, p_anova = f_oneway(*city_groups)

print("=" * 52)
print("🌆 Test 2: ANOVA — All City Prices Different?")
print("=" * 52)
print(f"   F-statistic : {f_stat:.4f}   p-value: {p_anova:.6f}")
print("   ✅ REJECT H₀ — At least one city has a different price" if p_anova < 0.05 else "   ✗ Fail")

In [ ]:
# Test 3: Chi-Square — Furnishing vs Status
ct = pd.crosstab(df['furnishing'], df['status'])
chi2, p_chi, dof, _ = chi2_contingency(ct)

print("=" * 52)
print("🔲 Test 3: Chi-Square — Furnishing vs Status")
print("=" * 52)
print(ct.to_string())
print(f"\n   Chi-square: {chi2:.4f}  p-value: {p_chi:.6f}  DoF: {dof}")
print("   ✅ REJECT H₀ — Furnishing type IS related to Status" if p_chi < 0.05 else "   ✗ Fail")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

city_data  = df[df['price_cr'] <= 15]
city_order = city_data.groupby('city')['price_cr'].median().sort_values(ascending=False).index

sns.violinplot(data=city_data, x='city', y='price_cr',
               order=city_order, palette=COLORS, ax=axes[0],
               inner='quartile', cut=0)
axes[0].set_title('Price by City (Violin — ANOVA p<0.05)')
axes[0].set_xlabel('City'); axes[0].set_ylabel('Price (₹ Cr)')
axes[0].tick_params(axis='x', rotation=45)

ct.plot(kind='bar', ax=axes[1], color=COLORS[:2], edgecolor='white')
axes[1].set_title('Furnishing vs Status (Chi-Square)')
axes[1].set_xlabel('Furnishing'); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

# PART 5 — Machine Learning Pipeline: Price Prediction

**Best Algorithm:** Gradient Boosting Regressor — ideal for tabular real estate data
- Handles mixed categorical/numerical features
- Robust to outliers (far better than linear models)
- No scaling required
- Naturally captures non-linear relationships (BHK × Area × City interactions)

**Pipeline Steps:**

| Step | Action |
|------|--------|
| 5.1 | ML Preprocessing — impute NaN, encode categoricals, log-transform target |
| 5.2 | Train-Test Split |
| 5.3 | Feature Selection (Filter · Wrapper · Embedded) |
| 5.4 | Apply Selected Features |
| 5.5 | Baseline Models (DT · Ridge · RF · GBM) |
| 5.6 | Hyperparameter Tuning (GridSearch · RandomSearch · Bayesian Optuna) |
| 5.7 | Cross Validation (KFold · RepeatedKFold) |
| 5.8 | Bias-Variance Tradeoff |
| 5.9 | Final Model Comparison |

## 5.1 ML Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# Features and target
feature_cols = ['city', 'bhk', 'area_sqft', 'furnishing', 'status']
target_col   = 'price_inr'

ml_df = df[feature_cols + [target_col]].copy()

print("NaN before imputation:")
print(ml_df.isnull().sum())

In [ ]:
# BHK: fill with mode (most common = 3 BHK)
ml_df['bhk'] = ml_df['bhk'].fillna(ml_df['bhk'].mode()[0])

# Categorical: fill with mode
for col in ['furnishing', 'status']:
    ml_df[col] = ml_df[col].fillna(ml_df[col].mode()[0])

print("NaN after imputation:", ml_df.isnull().sum().sum())

# Log-transform the target (price is right-skewed — log makes it ~normal)
y_log = np.log1p(ml_df[target_col])
X_raw = ml_df.drop(columns=[target_col])

print(f"\nX shape : {X_raw.shape} | y shape : {y_log.shape}")
print(f"Price range (Cr): ₹{ml_df[target_col].min()/1e7:.1f} – ₹{ml_df[target_col].max()/1e7:.1f}")

In [ ]:
# Encode categorical features with OrdinalEncoder
cat_cols = ['city', 'furnishing', 'status']
num_cols = ['bhk', 'area_sqft']

OE = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_encoded       = X_raw.copy()
X_encoded[cat_cols] = OE.fit_transform(X_raw[cat_cols])

print("Encoded feature sample:")
X_encoded.head()

## 5.2 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_log,
    test_size    = 0.2,
    random_state = 42
)

print(f"X_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"y_train : {y_train.shape}  |  y_test : {y_test.shape}")

## 5.3 Feature Selection

Methods applied:
- **SelectKBest** (f_regression) — ANOVA F-test for regression
- **Mutual Information** (mutual_info_regression) — information-theoretic
- **RFE** with LinearRegression — wrapper method
- **Feature Importances** from GBM — embedded method

In [ ]:
from sklearn.feature_selection import (SelectKBest, f_regression,
                                         mutual_info_regression, RFE)
from sklearn.linear_model import LinearRegression

# ── SelectKBest (F-test) ─────────────────────────────────────────────────────
kbest = SelectKBest(f_regression, k='all')
kbest.fit(X_train, y_train)

kbest_df = pd.DataFrame({
    'Feature': X_train.columns,
    'F_Score': kbest.scores_
}).sort_values('F_Score', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=kbest_df, x='Feature', y='F_Score', palette='Blues_r')
plt.title('SelectKBest — ANOVA F-Scores (Regression)')
plt.tight_layout(); plt.show()

print(kbest_df.to_string(index=False))

In [ ]:
# ── Mutual Information ────────────────────────────────────────────────────────
mi_scores = mutual_info_regression(X_train, y_train, random_state=42)

mi_df = pd.DataFrame({
    'Feature' : X_train.columns,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=mi_df, x='Feature', y='MI_Score', palette='Greens_r')
plt.title('Mutual Information Scores (higher = more predictive of price)')
plt.tight_layout(); plt.show()

print(mi_df.to_string(index=False))

In [ ]:
# ── RFE with LinearRegression ────────────────────────────────────────────────
rfe = RFE(estimator=LinearRegression(), n_features_to_select=4)
rfe.fit(X_train, y_train)

rfe_df = pd.DataFrame({
    'Feature' : X_train.columns,
    'Selected': rfe.support_,
    'Rank'    : rfe.ranking_
}).sort_values('Rank')

print("RFE Selected Features (rank=1):",
      X_train.columns[rfe.support_].tolist())
print()
print(rfe_df.to_string(index=False))

In [ ]:
# ── Embedded: GBM Feature Importances ────────────────────────────────────────
from sklearn.ensemble import GradientBoostingRegressor

gbm_tmp = GradientBoostingRegressor(n_estimators=100, random_state=42)
gbm_tmp.fit(X_train, y_train)

fi_df = pd.DataFrame({
    'Feature'   : X_train.columns,
    'Importance': gbm_tmp.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=fi_df, x='Feature', y='Importance', palette='Oranges_r')
plt.title('GBM Feature Importances (Embedded Method)')
plt.tight_layout(); plt.show()

print(fi_df.to_string(index=False))

In [ ]:
# ── Feature Selection Summary ─────────────────────────────────────────────────
from collections import Counter

kbest_top = set(kbest_df.head(4)['Feature'])
mi_top    = set(mi_df.head(4)['Feature'])
rfe_top   = set(X_train.columns[rfe.support_])
fi_top    = set(fi_df.head(4)['Feature'])

all_sel = list(kbest_top) + list(mi_top) + list(rfe_top) + list(fi_top)
votes   = Counter(all_sel)

summary_rows = []
for feat in X_train.columns:
    summary_rows.append({
        'Feature'    : feat,
        'SelectKBest': '✓' if feat in kbest_top else '✗',
        'MutualInfo' : '✓' if feat in mi_top    else '✗',
        'RFE'        : '✓' if feat in rfe_top   else '✗',
        'GBM_Importnc':'✓' if feat in fi_top    else '✗',
        'Votes'      : votes.get(feat, 0)
    })

summary_sel_df = pd.DataFrame(summary_rows).sort_values('Votes', ascending=False)
print(summary_sel_df.to_string(index=False))

## 5.4 Apply Selected Features

> Using features selected by **all 4 methods** (consensus voting ≥ 3).

In [ ]:
# Features voted by 3+ methods — most reliable selection
selected_features = [f for f, v in votes.items() if v >= 3]
if not selected_features:   # fallback: top 4 by MI
    selected_features = mi_df.head(4)['Feature'].tolist()

print(f"Selected {len(selected_features)} features: {selected_features}")

X_train_sel = X_train[selected_features]
X_test_sel  = X_test[selected_features]

print(f"X_train_sel : {X_train_sel.shape}")
print(f"X_test_sel  : {X_test_sel.shape}")

## 5.5 Baseline Models (Before Tuning)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred_log = model.predict(X_te)
    y_pred_inr = np.expm1(y_pred_log)
    y_test_inr = np.expm1(y_te)
    r2    = r2_score(y_te, y_pred_log)
    mae   = mean_absolute_error(y_test_inr, y_pred_inr) / 1e7    # in Crores
    rmse  = np.sqrt(mean_squared_error(y_test_inr, y_pred_inr)) / 1e7
    n, p  = X_te.shape
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return {'Model': name, 'R²': r2, 'Adj R²': adj_r2,
            'MAE (Cr)': mae, 'RMSE (Cr)': rmse, '_model': model}

baselines = [
    evaluate('DT Regressor',  DecisionTreeRegressor(random_state=42),
             X_train_sel, y_train, X_test_sel, y_test),
    evaluate('Ridge',          Ridge(alpha=1.0),
             X_train_sel, y_train, X_test_sel, y_test),
    evaluate('Random Forest',  RandomForestRegressor(n_estimators=100, random_state=42),
             X_train_sel, y_train, X_test_sel, y_test),
    evaluate('GBM (baseline)', GradientBoostingRegressor(n_estimators=100, random_state=42),
             X_train_sel, y_train, X_test_sel, y_test),
]

base_df = pd.DataFrame([{k:v for k,v in b.items() if k!='_model'} for b in baselines])
base_df = base_df.sort_values('R²', ascending=False)
base_df = base_df.round(4)
print(base_df.to_string(index=False))

## 5.6 Hyperparameter Tuning

> Tuning **Gradient Boosting Regressor** — the best baseline algorithm.

### 5.6.1 GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [3, 4, 5],
    'learning_rate'    : [0.05, 0.1],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator  = GradientBoostingRegressor(random_state=42),
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'r2',
    n_jobs     = -1,
    verbose    = 1
)

grid_search.fit(X_train_sel, y_train)

print("Best Parameters :", grid_search.best_params_)
print("Best R² (CV)    :", round(grid_search.best_score_, 4))

GBM_grid = grid_search.best_estimator_

### 5.6.2 RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators'     : [50, 100, 150, 200, 300],
    'max_depth'        : [2, 3, 4, 5, 6],
    'learning_rate'    : [0.01, 0.05, 0.1, 0.15, 0.2],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'subsample'        : [0.7, 0.8, 0.9, 1.0]
}

random_search = RandomizedSearchCV(
    estimator           = GradientBoostingRegressor(random_state=42),
    param_distributions = param_dist,
    n_iter              = 40,
    cv                  = 5,
    scoring             = 'r2',
    random_state        = 42,
    n_jobs              = -1,
    verbose             = 1
)

random_search.fit(X_train_sel, y_train)

print("Best Parameters :", random_search.best_params_)
print("Best R² (CV)    :", round(random_search.best_score_, 4))

GBM_random = random_search.best_estimator_

### 5.6.3 Bayesian Optimization — Optuna

In [ ]:
import optuna
from sklearn.model_selection import KFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Create Objective Function
def Objective(trial):
    # create search space
    n_estimators      = trial.suggest_int('n_estimators', 50, 300)
    max_depth         = trial.suggest_int('max_depth', 2, 8)
    learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 8)
    subsample         = trial.suggest_float('subsample', 0.6, 1.0)

    # train the algorithm
    gbm = GradientBoostingRegressor(
        n_estimators      = n_estimators,
        max_depth         = max_depth,
        learning_rate     = learning_rate,
        min_samples_split = min_samples_split,
        min_samples_leaf  = min_samples_leaf,
        subsample         = subsample,
        random_state      = 42
    )

    # get the Performance metrics
    kf    = KFold(n_splits=5, shuffle=True, random_state=23)
    score = cross_val_score(estimator=gbm, X=X_train_sel, y=y_train,
                            scoring='r2', cv=kf).mean()
    return score

# 2. Create the Study
study = optuna.create_study(direction='maximize')

# 3. Evaluate model Performance
study.optimize(Objective, n_trials=40, show_progress_bar=True)

In [ ]:
print("Best Parameters :", study.best_params)
print("Best R² (CV)    :", round(study.best_value, 4))

GBM_bayesian = GradientBoostingRegressor(**study.best_params, random_state=42)
GBM_bayesian.fit(X_train_sel, y_train)

## 5.7 Cross Validation

Applied on **GBM_bayesian** (best tuned model) using **X_train_sel**.
Scoring: `r2` — proportion of variance explained.

| Strategy | Description |
|---|---|
| KFold-5 | 5 equally-sized folds |
| KFold-10 | 10 folds — lower bias estimate |
| RepeatedKFold | 5 splits × 3 repeats = 15 evaluations — more stable |

### 7.1 K-Fold (5 splits)

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

k5 = KFold(n_splits=5, shuffle=True, random_state=44)
sc_k5 = cross_val_score(GBM_bayesian, X_train_sel, y_train, cv=k5, scoring='r2')

print("KFold-5  per fold :", sc_k5.round(4))
print(f"Mean R² : {sc_k5.mean():.4f}  |  Std : {sc_k5.std():.4f}")

### 7.2 K-Fold (10 splits)

In [ ]:
k10   = KFold(n_splits=10, shuffle=True, random_state=44)
sc_k10 = cross_val_score(GBM_bayesian, X_train_sel, y_train, cv=k10, scoring='r2')

print("KFold-10 per fold :", sc_k10.round(4))
print(f"Mean R² : {sc_k10.mean():.4f}  |  Std : {sc_k10.std():.4f}")

### 7.3 RepeatedKFold (5 splits × 3 repeats)

In [ ]:
from sklearn.model_selection import RepeatedKFold

rkf    = RepeatedKFold(n_splits=5, n_repeats=3, random_state=44)
sc_rkf = cross_val_score(GBM_bayesian, X_train_sel, y_train, cv=rkf, scoring='r2')

print("RepeatedKFold (15 evaluations):")
print("Scores :", sc_rkf.round(4))
print(f"Mean R² : {sc_rkf.mean():.4f}  |  Std : {sc_rkf.std():.4f}")

### 7.4 Cross Validation Summary

In [ ]:
cv_summary = pd.DataFrame({
    'Strategy'  : ['KFold-5', 'KFold-10', 'RepeatedKFold(5×3)'],
    'Folds'     : [5, 10, 15],
    'Mean R²'   : [sc_k5.mean(), sc_k10.mean(), sc_rkf.mean()],
    'Std R²'    : [sc_k5.std(),  sc_k10.std(),  sc_rkf.std()]
}).round(4)

print(cv_summary.to_string(index=False))

## 5.8 Bias-Variance Tradeoff

- **High Bias** → underfitting: both train & val scores are low
- **High Variance** → overfitting: train score high, val score much lower
- **Sweet spot** → train and val scores converge at a good level

### 8.1 Learning Curves — sklearn

In [ ]:
from sklearn.model_selection import learning_curve

gbm_lc = GradientBoostingRegressor(**study.best_params, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    estimator    = gbm_lc,
    X            = X_train_sel,
    y            = y_train,
    cv           = 5,
    scoring      = 'r2',
    train_sizes  = np.linspace(0.1, 1.0, 10),
    n_jobs       = -1,
    random_state = 42
)

train_mean = train_scores.mean(axis=1);  train_std = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1);    val_std   = val_scores.std(axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='royalblue', label='Training R²')
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std,
                 alpha=0.15, color='royalblue')
plt.plot(train_sizes, val_mean, 'o-', color='seagreen', label='Validation R²')
plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std,
                 alpha=0.15, color='seagreen')
plt.xlabel('Training Samples')
plt.ylabel('R² Score')
plt.title('Learning Curves — GBM Bayesian Best Model\n(Bias-Variance Tradeoff)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 8.2 Bias-Variance Decomposition — mlxtend

In [ ]:
from mlxtend.evaluate import bias_variance_decomp

gbm_bv = GradientBoostingRegressor(**study.best_params, random_state=42)

avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
    gbm_bv,
    X_train_sel.values,
    y_train.values,
    X_test_sel.values,
    y_test.values,
    loss        = 'mse',
    random_seed = 42
)

print('=' * 48)
print('  Bias-Variance Decomposition — GBM Bayesian')
print('=' * 48)
print(f'  Average Expected Loss : {avg_expected_loss:.6f}')
print(f'  Average Bias²         : {avg_bias:.6f}')
print(f'  Average Variance      : {avg_var:.6f}')
print('-' * 48)
print(f'  Bias² + Variance      : {avg_bias + avg_var:.6f}')
print('=' * 48)

### 8.3 Learning Curves — mlxtend

In [ ]:
from mlxtend.plotting import plot_learning_curves

gbm_lc2 = GradientBoostingRegressor(**study.best_params, random_state=42)

plot_learning_curves(
    X_train_sel.values,
    y_train.values,
    X_test_sel.values,
    y_test.values,
    clf = gbm_lc2
)
plt.title('Learning Curves — GBM Bayesian (mlxtend)')
plt.tight_layout()
plt.show()

## 5.9 Final Model Comparison

In [ ]:
all_models = {
    'DT Regressor'  : baselines[0]['_model'],
    'Ridge'         : baselines[1]['_model'],
    'Random Forest' : baselines[2]['_model'],
    'GBM Baseline'  : baselines[3]['_model'],
    'GBM GridSearch': GBM_grid,
    'GBM RandomSrch': GBM_random,
    'GBM Bayesian'  : GBM_bayesian
}

results = []

print(f"{'Model':<18} {'R²':>8} {'Adj R²':>10} {'MAE (Cr)':>12} {'RMSE (Cr)':>12}")
print('-' * 64)

n, p = X_test_sel.shape

for name, model in all_models.items():
    y_pred_log = model.predict(X_test_sel)
    y_pred_inr = np.expm1(y_pred_log)
    y_test_inr = np.expm1(y_test)

    r2     = r2_score(y_test, y_pred_log)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    mae    = mean_absolute_error(y_test_inr, y_pred_inr) / 1e7
    rmse   = np.sqrt(mean_squared_error(y_test_inr, y_pred_inr)) / 1e7

    results.append({'Model': name, 'R²': r2, 'Adj R²': adj_r2,
                    'MAE (Cr)': mae, 'RMSE (Cr)': rmse})

    print(f"{name:<18} {r2:>8.4f} {adj_r2:>10.4f} {mae:>12.4f} {rmse:>12.4f}")

In [ ]:
results_df = pd.DataFrame(results).sort_values('R²', ascending=False)

print("\n=== Best Model ===")
best = results_df.iloc[0]
print(f"Model     : {best['Model']}")
print(f"R²        : {best['R²']:.4f}")
print(f"Adj R²    : {best['Adj R²']:.4f}")
print(f"MAE       : ₹{best['MAE (Cr)']:.2f} Cr")
print(f"RMSE      : ₹{best['RMSE (Cr)']:.2f} Cr")

In [ ]:
# R² comparison bar chart
plt.figure(figsize=(11, 5))
colors_bar = [COLORS[0] if 'Bayesian' in m else COLORS[1] if 'GBM' in m
              else COLORS[2] if 'Forest' in m else '#cccccc'
              for m in results_df['Model']]

bars = plt.bar(results_df['Model'], results_df['R²'], color=colors_bar, edgecolor='white')
plt.axhline(0, color='black', lw=0.8)
for bar, val in zip(bars, results_df['R²']):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.ylabel('R² Score')
plt.title('Model Comparison — R² on Test Set (higher = better)')
plt.xticks(rotation=30, ha='right')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted for best model (GBM Bayesian)
y_pred_best    = np.expm1(GBM_bayesian.predict(X_test_sel))
y_actual       = np.expm1(y_test)
y_actual_cr    = y_actual / 1e7
y_pred_best_cr = y_pred_best / 1e7

plt.figure(figsize=(8, 6))
plt.scatter(y_actual_cr, y_pred_best_cr, alpha=0.3, color=COLORS[0], s=15)
lims = [min(y_actual_cr.min(), y_pred_best_cr.min()),
        max(y_actual_cr.quantile(0.99), y_pred_best_cr.max() * 0.8)]
plt.plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Price (₹ Crores)')
plt.ylabel('Predicted Price (₹ Crores)')
plt.title('Actual vs Predicted — GBM Bayesian Best Model')
plt.legend()
plt.xlim(0, 10); plt.ylim(0, 10)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance of best model
fi_best = pd.DataFrame({
    'Feature'   : selected_features,
    'Importance': GBM_bayesian.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=fi_best, x='Feature', y='Importance', palette='Reds_r')
plt.title('Feature Importances — GBM Bayesian Final Model')
plt.tight_layout()
plt.show()

print(fi_best.to_string(index=False))